# 04 · Feature Engineering + LightGBM + Optuna

Pipeline completo de preprocesamiento, feature engineering, entrenamiento con walk-forward y optimización bayesiana de hiperparámetros.

## Estructura
1. Setup y carga del dataset procesado
2. Preprocesamiento (interpolación de nulos aislados)
3. Feature engineering (lags, deltas, ratios, tendencias)
4. Modelo baseline LightGBM + evaluación por ganancia
5. Optimización de hiperparámetros con Optuna
6. Evaluación walk-forward completa
7. Predicción final sobre 202108 + archivo de submission

---
**Proyecto:** Predicción de Churn Bancario · Data Mining en EyF  
**Etapa:** 4 — Modelado  
**Última actualización:** 2026-10

## 0 · Setup

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
!pip install optuna lightgbm --quiet

In [5]:
import gc
import warnings
import json
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import lightgbm as lgb
import optuna
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_param_importances,
    plot_contour,
)
from sklearn.metrics import roc_auc_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110

# ─── Rutas ────────────────────────────────────────────────────────────────────
RUTA_BASE  = Path('/content/drive/MyDrive/Maestría Data Mining/DM en EyF')
RUTA_COMP  = RUTA_BASE / 'Competencia 1'
RUTA_COMP.mkdir(parents=True, exist_ok=True)

ARCHIVO_PARQUET    = RUTA_COMP / 'dataset_target_construido.parquet'
ARCHIVO_SUBMISSION = RUTA_COMP / f'submission_{datetime.now().strftime("%Y%m%d_%H%M")}.csv'
ARCHIVO_METRICAS   = RUTA_COMP / 'metricas_experimentos.csv'
ARCHIVO_PARAMS     = RUTA_COMP / 'mejores_params_optuna.json'

# ─── Constantes ───────────────────────────────────────────────────────────────
COL_ID      = 'numero_de_cliente'
COL_PERIODO = 'foto_mes'
TARGET_RAW  = 'clase_ternaria'
TARGET      = 'es_baja2'

MESES_CON_TARGET = [202103, 202104, 202105, 202106]
MES_PREDECIR     = 202108   # sin target — a predecir en la competencia
MES_TRAIN_EXTRA  = 202107   # sin target — puede usarse para entrenar el modelo final

# Función de ganancia
GANANCIA_TP =  1_072_500
COSTO_FP    =    -27_500
RATIO_BREAK_EVEN = abs(GANANCIA_TP / COSTO_FP)  # 39x

# Semilla global para reproducibilidad
SEED = 42

print(f'Archivo input: {ARCHIVO_PARQUET.exists()}')
print(f'RATIO_BREAK_EVEN: {RATIO_BREAK_EVEN:.0f}x')

Archivo input: True
RATIO_BREAK_EVEN: 39x


## 1 · Funciones de ganancia (centralizadas)

Todas las evaluaciones del proyecto pasan por estas funciones.

In [6]:
def calcular_ganancia(y_true: np.ndarray, y_pred_bin: np.ndarray) -> float:
    """Ganancia total dados arrays de target real y predicción binaria."""
    tp = ((y_true == 1) & (y_pred_bin == 1)).sum()
    fp = ((y_true == 0) & (y_pred_bin == 1)).sum()
    return tp * GANANCIA_TP + fp * COSTO_FP


def ganancia_por_threshold(y_true: np.ndarray, y_prob: np.ndarray,
                            n_thresholds: int = 500) -> pd.DataFrame:
    """Barre thresholds y devuelve ganancia, TP, FP y N predichos para cada uno."""
    thresholds = np.linspace(0.001, 0.999, n_thresholds)
    rows = []
    for t in thresholds:
        pred = (y_prob >= t).astype(int)
        tp = ((y_true == 1) & (pred == 1)).sum()
        fp = ((y_true == 0) & (pred == 1)).sum()
        rows.append({'threshold': t,
                     'ganancia': tp * GANANCIA_TP + fp * COSTO_FP,
                     'tp': int(tp), 'fp': int(fp),
                     'n_intervenidos': int(tp + fp)})
    return pd.DataFrame(rows)


def threshold_optimo(y_true: np.ndarray, y_prob: np.ndarray) -> dict:
    """Devuelve el threshold que maximiza la ganancia y las métricas asociadas."""
    df = ganancia_por_threshold(y_true, y_prob)
    idx = df['ganancia'].idxmax()
    return df.loc[idx].to_dict()


def lgbm_ganancia_metric(y_pred: np.ndarray, dataset: lgb.Dataset):
    """Métrica custom para LightGBM: ganancia en el threshold óptimo.
    Se usa como eval_metric en el entrenamiento — higher is better.
    """
    y_true = dataset.get_label().astype(int)
    opt    = threshold_optimo(y_true, y_pred)
    return 'ganancia', opt['ganancia'], True   # name, value, higher_is_better


def plot_ganancia_threshold(y_true: np.ndarray, y_prob: np.ndarray,
                             titulo: str = '') -> dict:
    """Grafica ganancia y N intervenidos vs threshold. Retorna el óptimo."""
    df = ganancia_por_threshold(y_true, y_prob)
    opt = df.loc[df['ganancia'].idxmax()].to_dict()

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # Ganancia
    ax = axes[0]
    ax.plot(df['threshold'], df['ganancia'] / 1e9, color='#4C72B0', lw=2)
    ax.axvline(opt['threshold'], color='#C44E52', ls='--', lw=1.5,
               label=f"Óptimo: {opt['threshold']:.4f}")
    ax.axhline(0, color='gray', ls=':', lw=1)
    ax.set_title(f'Ganancia vs Threshold {titulo}', fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('Ganancia (miles de millones $)')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.1f}B'))
    ax.legend(fontsize=9)

    # Intervenidos
    ax = axes[1]
    ax.plot(df['threshold'], df['n_intervenidos'], color='#DD8452', lw=2)
    ax.axvline(opt['threshold'], color='#C44E52', ls='--', lw=1.5)
    ax.set_title(f'Clientes a intervenir vs Threshold {titulo}', fontweight='bold')
    ax.set_xlabel('Threshold')
    ax.set_ylabel('N clientes')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{int(x):,}'))

    plt.tight_layout()
    plt.show()

    print(f"  Threshold óptimo:  {opt['threshold']:.5f}")
    print(f"  Ganancia:          ${opt['ganancia']:>15,.0f}")
    print(f"  TP (detectados):   {opt['tp']:,}")
    print(f"  FP (en vano):      {opt['fp']:,}")
    print(f"  Total intervenidos:{opt['n_intervenidos']:,}")
    return opt


print('✓ Funciones de ganancia cargadas.')

✓ Funciones de ganancia cargadas.


## 2 · Carga de datos

In [7]:
df = pd.read_parquet(ARCHIVO_PARQUET)
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f'Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas — {mem_mb:.1f} MB')
print(f'Meses: {sorted(df[COL_PERIODO].unique())}')
print(f'Target: {df[TARGET].value_counts(dropna=False).to_dict()}')

Dataset cargado: 983,061 filas × 156 columnas — 411.6 MB
Meses: [np.uint32(202103), np.uint32(202104), np.uint32(202105), np.uint32(202106), np.uint32(202107), np.uint32(202108)]
Target: {0.0: 645999, nan: 332995, 1.0: 4067}


## 3 · Preprocesamiento — Interpolación de nulos aislados

Rellenamos NaN en variables numéricas continuas **solo si el cliente tiene dato en el mes anterior Y el posterior** (nulo aislado). Esto es correcto para variables tipo saldo, rentabilidad, etc., pero **no** para flags binarios ni conteos que pueden ser legítimamente 0.

Para la competencia, el NaN en el último mes que queremos predecir (202108) **no se interpola** — ese mes no tiene un mes posterior conocido.

In [8]:
# Variables que NO interpolamos:
# - IDs, período, targets
# - Flags binarios (active_quarter, cliente_vip, internet, tcallcenter, thomebanking, tmobile_app)
# - Conteos enteros (c*: cantidad de productos/transacciones) — pueden ser 0 de forma legítima
# - Columnas de estado categórico (Master_status, Visa_status, Master_delinquency, etc.)

NO_INTERPOLAR = {
    COL_ID, COL_PERIODO, TARGET_RAW, TARGET,
    'active_quarter', 'cliente_vip', 'internet',
    'tcallcenter', 'thomebanking', 'tmobile_app',
    'Master_status', 'Visa_status',
    'Master_delinquency', 'Visa_delinquency',
}

# Conteos: columnas que empiezan con 'c' (cantidad) o son flags de canal (t*)
NO_INTERPOLAR.update([c for c in df.columns if c.startswith('c') and df[c].dtype in ['uint8', 'uint16', 'uint32', 'int8', 'int16', 'int32']])

# Columnas a interpolar: numéricas continuas (float) que NO están en NO_INTERPOLAR
COLS_INTERPOLAR = [
    c for c in df.select_dtypes(include=['float32', 'float64']).columns
    if c not in NO_INTERPOLAR
]

print(f'Columnas a interpolar: {len(COLS_INTERPOLAR)}')
print(f'Columnas excluidas de interpolación: {len(NO_INTERPOLAR)}')

# NaN antes
nulos_antes = df[COLS_INTERPOLAR].isna().sum().sum()
print(f'\nNaN antes de interpolación: {nulos_antes:,}')

Columnas a interpolar: 85
Columnas excluidas de interpolación: 70

NaN antes de interpolación: 9,832,054


In [ ]:
# Interpolación lineal dentro de cada cliente, limit=1 (solo NaN aislados)
# Ordenamos por cliente y mes para que la interpolación sea correcta en el tiempo
df = df.sort_values([COL_ID, COL_PERIODO]).reset_index(drop=True)

df[COLS_INTERPOLAR] = (
    df.groupby(COL_ID)[COLS_INTERPOLAR]
    .transform(lambda s: s.interpolate(
        method='linear',
        limit=1,             # solo 1 NaN consecutivo
        limit_direction='both'  # puede ser NaN al inicio o al final del grupo (pero limit=1 aplica)
    ))
)

nulos_despues = df[COLS_INTERPOLAR].isna().sum().sum()
print(f'NaN después de interpolación: {nulos_despues:,}')
print(f'NaN rellenados: {nulos_antes - nulos_despues:,}')
gc.collect()

## 4 · Feature Engineering

Construimos features en cinco familias, de mayor a menor importancia esperada en datos bancarios de panel:

1. **Lag (t-1)** — valor del mes anterior: captura el comportamiento reciente
2. **Delta (t - t-1)** — variación absoluta: captura tendencia
3. **Delta relativo (delta / |t-1|)** — variación proporcional: más informativa para variables con diferente escala
4. **Tendencia corta** — pendiente de regresión lineal sobre ventana de 3 meses: deterioro progresivo
5. **Ratios entre variables** — estructura del portfolio del cliente
6. **Flags** — primer mes en el panel, tiene tarjeta, etc.

In [ ]:
# Variables base para FE — excluimos targets, IDs, período y las ya excluidas
EXCLUIR_FE = {COL_ID, COL_PERIODO, TARGET_RAW, TARGET}

# Seleccionamos variables numéricas continuas con varianza suficiente
# para no generar miles de features inútiles
cols_base_fe = [
    c for c in df.select_dtypes(include=[np.number]).columns
    if c not in EXCLUIR_FE
    and df[c].std() > 0
]

print(f'Variables base para FE: {len(cols_base_fe)}')
print(f'Columnas totales antes del FE: {df.shape[1]}')

In [ ]:
# ── 4.1 Lag y Delta ────────────────────────────────────────────────────────────
# Generamos lag(t-1) y delta para cada variable base
# Usamos groupby + shift para respetar la estructura de panel

print('Generando lags y deltas...')
nuevas_cols = []

# Trabajamos sobre el df ya ordenado por (cliente, mes)
for col in cols_base_fe:
    lag_col  = f'{col}_lag1'
    delta_col = f'{col}_delta1'

    df[lag_col]   = df.groupby(COL_ID)[col].shift(1)
    df[delta_col] = df[col] - df[lag_col]

    nuevas_cols.extend([lag_col, delta_col])

gc.collect()
print(f'  Features generadas: {len(nuevas_cols):,}')
print(f'  Columnas totales: {df.shape[1]:,}')

In [ ]:
# ── 4.2 Delta relativo (ratio de cambio) ──────────────────────────────────────
# Solo para variables donde el lag no es 0 (evitar división por cero)
# Captura variaciones proporcionales — más informativo para variables de distinta escala

print('Generando deltas relativos...')
for col in cols_base_fe:
    lag_col   = f'{col}_lag1'
    drel_col  = f'{col}_drel1'
    lag_vals  = df[lag_col].replace(0, np.nan)  # evitar / 0
    df[drel_col] = df[f'{col}_delta1'] / lag_vals.abs()
    # Clipeamos outliers extremos del ratio (±10 es suficiente para la señal)
    df[drel_col] = df[drel_col].clip(-10, 10)
    nuevas_cols.append(drel_col)

gc.collect()
print(f'  Features totales: {len(nuevas_cols):,}')
print(f'  Columnas totales: {df.shape[1]:,}')

In [ ]:
# ── 4.3 Tendencia de 3 meses (ventana corta) ──────────────────────────────────
# Pendiente de regresión lineal sobre [t-2, t-1, t] para variables clave
# Solo para variables de alto valor predictivo (montos principales)
# para no explotar RAM con 150 variables × tendencia

COLS_TENDENCIA = [
    'mrentabilidad', 'mrentabilidad_annual', 'mcuentas_saldo',
    'mcuenta_corriente', 'mcaja_ahorro', 'mcomisiones',
    'mtarjeta_visa_consumo', 'mtarjeta_master_consumo',
    'mprestamos_personales', 'mprestamos_hipotecarios',
    'cproductos', 'chomebanking_transacciones', 'ctrx_quarter',
    'mactivos_margen', 'mpasivos_margen',
]
# Filtramos solo las que existen en el df
COLS_TENDENCIA = [c for c in COLS_TENDENCIA if c in df.columns]

print(f'Generando tendencias para {len(COLS_TENDENCIA)} variables...')

def pendiente_3(s: pd.Series) -> pd.Series:
    """Pendiente de regresión lineal sobre ventana rodante de 3 obs."""
    def slope(w):
        if w.isna().any() or len(w) < 3:
            return np.nan
        x = np.arange(len(w))
        return np.polyfit(x, w, 1)[0]
    return s.rolling(window=3, min_periods=3).apply(slope, raw=True)

for col in COLS_TENDENCIA:
    tend_col = f'{col}_tend3'
    df[tend_col] = df.groupby(COL_ID)[col].transform(pendiente_3)
    nuevas_cols.append(tend_col)

gc.collect()
print(f'  Features totales: {len(nuevas_cols):,}')
print(f'  Columnas totales: {df.shape[1]:,}')

In [ ]:
# ── 4.4 Ratios entre variables (estructura del portfolio) ──────────────────────
# Literatura bancaria: la *estructura relativa* del portfolio es más predictiva
# que los valores absolutos, especialmente en contextos inflacionarios.

print('Generando ratios...')

# Seguridad: evitamos división por cero reemplazando 0 con NaN
def safe_ratio(a: pd.Series, b: pd.Series, clip: float = 100) -> pd.Series:
    return (a / b.replace(0, np.nan)).clip(-clip, clip)

# Rentabilidad sobre activos
df['r_rentab_activos'] = safe_ratio(df['mrentabilidad'], df['mactivos_margen'].abs())

# Caja ahorro sobre saldo total de cuentas
df['r_ahorro_saldo']   = safe_ratio(df['mcaja_ahorro'], df['mcuentas_saldo'].abs())

# Comisiones sobre rentabilidad
df['r_comis_rentab']   = safe_ratio(df['mcomisiones'], df['mrentabilidad'].abs())

# Consumo visa sobre limite visa (utilización)
if 'mtarjeta_visa_consumo' in df.columns and 'Visa_mlimitecompra' in df.columns:
    df['r_visa_utilizacion'] = safe_ratio(df['mtarjeta_visa_consumo'], df['Visa_mlimitecompra'])

# Consumo master sobre limite master
if 'mtarjeta_master_consumo' in df.columns and 'Master_mlimitecompra' in df.columns:
    df['r_master_utilizacion'] = safe_ratio(df['mtarjeta_master_consumo'], df['Master_mlimitecompra'])

# Deuda total sobre ingresos (payroll)
deuda_total = (
    df.get('mprestamos_personales', 0).fillna(0) +
    df.get('mprestamos_hipotecarios', 0).fillna(0) +
    df.get('mprestamos_prendarios', 0).fillna(0)
)
df['r_deuda_payroll'] = safe_ratio(deuda_total, df['mpayroll'].abs())

# Transacciones digitales vs. total (digitalización)
trx_digital = (
    df.get('chomebanking_transacciones', pd.Series(0, index=df.index)).fillna(0) +
    df.get('cmobile_app_trx', pd.Series(0, index=df.index)).fillna(0)
)
trx_total = df.get('ctrx_quarter', pd.Series(1, index=df.index)).replace(0, np.nan)
df['r_digitalizacion'] = (trx_digital / trx_total).clip(0, 1)

ratios_nuevos = [
    'r_rentab_activos', 'r_ahorro_saldo', 'r_comis_rentab',
    'r_visa_utilizacion', 'r_master_utilizacion',
    'r_deuda_payroll', 'r_digitalizacion'
]
ratios_nuevos = [c for c in ratios_nuevos if c in df.columns]
nuevas_cols.extend(ratios_nuevos)

gc.collect()
print(f'  Ratios generados: {len(ratios_nuevos)}')
print(f'  Features totales: {len(nuevas_cols):,}')

In [ ]:
# ── 4.5 Flags ─────────────────────────────────────────────────────────────────

# Flag: es el primer mes del cliente en el panel (no tiene lags reales)
df['flag_primer_mes'] = (
    df.groupby(COL_ID)[COL_PERIODO].transform('min') == df[COL_PERIODO]
).astype('int8')

# Flag: tiene tarjeta Visa
if 'Visa_msaldototal' in df.columns:
    df['flag_tiene_visa']   = df['Visa_msaldototal'].notna().astype('int8')

# Flag: tiene tarjeta Master
if 'Master_msaldototal' in df.columns:
    df['flag_tiene_master'] = df['Master_msaldototal'].notna().astype('int8')

# Flag: tiene préstamo activo
df['flag_tiene_prestamo'] = (
    (df.get('cprestamos_personales',   pd.Series(0, index=df.index)).fillna(0) +
     df.get('cprestamos_hipotecarios', pd.Series(0, index=df.index)).fillna(0) +
     df.get('cprestamos_prendarios',   pd.Series(0, index=df.index)).fillna(0)) > 0
).astype('int8')

# Antigüedad en meses (proxy de estabilidad)
if 'cliente_antiguedad' in df.columns:
    df['antiguedad_bucket'] = pd.cut(
        df['cliente_antiguedad'],
        bins=[0, 12, 36, 60, 120, 9999],
        labels=[0, 1, 2, 3, 4]
    ).astype('int8')

flags_nuevos = ['flag_primer_mes', 'flag_tiene_visa', 'flag_tiene_master',
                'flag_tiene_prestamo', 'antiguedad_bucket']
flags_nuevos = [c for c in flags_nuevos if c in df.columns]
nuevas_cols.extend(flags_nuevos)

gc.collect()
print(f'Flags generados: {len(flags_nuevos)}')
print(f'TOTAL features nuevas: {len(nuevas_cols):,}')
print(f'Columnas totales en df: {df.shape[1]:,}')
mem_mb = df.memory_usage(deep=True).sum() / 1024**2
print(f'Memoria actual: {mem_mb:.1f} MB')

## 5 · Preparación de splits para walk-forward

Definimos los 3 folds utilizables y el split final para predecir `202108`.

In [ ]:
# Columnas a usar como features (todo excepto IDs, períodos, targets)
EXCLUIR_FEATURES = {COL_ID, COL_PERIODO, TARGET_RAW, TARGET}
FEATURE_COLS = [
    c for c in df.columns
    if c not in EXCLUIR_FEATURES
    and df[c].dtype != 'category'  # category no va directo a lgbm sin encoding
    or c in ['antiguedad_bucket']   # esta sí la queremos aunque sea category
]
# Limpiamos duplicados
FEATURE_COLS = list(dict.fromkeys(FEATURE_COLS))
print(f'Features para el modelo: {len(FEATURE_COLS)}')

# Folds walk-forward: (meses_train, mes_test)
FOLDS = [
    {'nombre': 'fold_2', 'train': [202103],               'test': 202104},
    {'nombre': 'fold_3', 'train': [202103, 202104],       'test': 202105},
    {'nombre': 'fold_4', 'train': [202103, 202104, 202105], 'test': 202106},
]

def get_split(df, meses_train, mes_test, feature_cols, target_col):
    """Extrae X_train, y_train, X_test, y_test para un fold walk-forward.
    Excluye filas con target NaN (baja1 y meses sin target).
    """
    mask_train = df[COL_PERIODO].isin(meses_train) & df[target_col].notna()
    mask_test  = (df[COL_PERIODO] == mes_test)      & df[target_col].notna()

    X_train = df.loc[mask_train, feature_cols]
    y_train = df.loc[mask_train, target_col].astype(int)
    X_test  = df.loc[mask_test,  feature_cols]
    y_test  = df.loc[mask_test,  target_col].astype(int)

    return X_train, y_train, X_test, y_test

# Verificamos tamaños
for fold in FOLDS:
    Xtr, ytr, Xte, yte = get_split(df, fold['train'], fold['test'], FEATURE_COLS, TARGET)
    n_pos_tr = ytr.sum()
    n_pos_te = yte.sum()
    print(f"{fold['nombre']}: train={len(ytr):,} (baja2={n_pos_tr}, {n_pos_tr/len(ytr)*100:.2f}%) | "
          f"test={len(yte):,} (baja2={n_pos_te}, {n_pos_te/len(yte)*100:.2f}%)")

## 6 · Baseline LightGBM

Entrenamos con parámetros default sobre el fold más representativo (fold_4) para:
- Establecer una ganancia de referencia
- Ver la feature importance antes de optimizar
- Entender el rango de thresholds óptimos

**Nota sobre `scale_pos_weight`:** con 160:1 de imbalance, `is_unbalance=True` es más simple pero puede descalibrar las probabilidades. Preferimos `scale_pos_weight` explícito para tener control total sobre el threshold — esto es crítico dado que buscamos maximizar ganancia, no AUC.

In [ ]:
# Ratio de imbalance para scale_pos_weight
# Usamos todos los meses entrenables para calcularlo
mask_total_train = df[COL_PERIODO].isin(MESES_CON_TARGET) & df[TARGET].notna()
y_total = df.loc[mask_total_train, TARGET]
SCALE_POS_WEIGHT = (y_total == 0).sum() / (y_total == 1).sum()
print(f'scale_pos_weight (ratio neg/pos): {SCALE_POS_WEIGHT:.1f}')

PARAMS_BASE = {
    'objective':        'binary',
    'metric':           'binary_logloss',
    'boosting_type':    'gbdt',
    'num_leaves':       31,
    'learning_rate':    0.05,
    'n_estimators':     500,
    'max_depth':        -1,
    'min_child_samples': 20,
    'subsample':        0.8,
    'subsample_freq':   1,
    'colsample_bytree': 0.8,
    'reg_alpha':        0.0,
    'reg_lambda':       0.0,
    'scale_pos_weight': SCALE_POS_WEIGHT,
    'random_state':     SEED,
    'verbose':         -1,
    'n_jobs':          -1,
}

print('\nEntrenando baseline sobre fold_4...')
Xtr, ytr, Xte, yte = get_split(df, [202103, 202104, 202105], 202106, FEATURE_COLS, TARGET)

modelo_base = lgb.LGBMClassifier(**PARAMS_BASE)
modelo_base.fit(
    Xtr, ytr,
    eval_set=[(Xte, yte)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)

prob_base = modelo_base.predict_proba(Xte)[:, 1]
auc_base  = roc_auc_score(yte, prob_base)
print(f'AUC baseline: {auc_base:.4f}')

In [ ]:
print('=== BASELINE — fold_4 (test = 202106) ===')
opt_base = plot_ganancia_threshold(yte.values, prob_base, titulo='(baseline)')

In [ ]:
# Feature importance baseline
fi = pd.DataFrame({
    'feature':    modelo_base.feature_name_,
    'importance': modelo_base.feature_importances_
}).sort_values('importance', ascending=False)

top_n = 30
fig, ax = plt.subplots(figsize=(8, 9))
fi.head(top_n).sort_values('importance').plot(
    kind='barh', x='feature', y='importance', ax=ax,
    color='#4C72B0', edgecolor='white', legend=False
)
ax.set_title(f'Top {top_n} features por importancia — baseline', fontweight='bold')
ax.set_xlabel('Importancia (split)')
plt.tight_layout()
plt.show()

print(f'\nTop 10 features:')
print(fi.head(10).to_string(index=False))

## 7 · Optimización de hiperparámetros con Optuna

### Hiperparámetros a optimizar y por qué

Basado en la literatura de LightGBM para clasificación binaria con imbalance extremo (160:1), los hiperparámetros de mayor impacto son, en orden:

| Parámetro | Rango | Motivo |
|---|---|---|
| `num_leaves` | 16–256 | Principal control de complejidad del árbol — más importante que max_depth |
| `learning_rate` | 0.01–0.3 (log) | Velocidad de aprendizaje — interactúa con n_estimators |
| `min_child_samples` | 5–200 | Regularización de hoja — crítico con datos muy desbalanceados |
| `feature_fraction` | 0.4–1.0 | % de features por árbol — reduce overfitting y correlación |
| `bagging_fraction` | 0.4–1.0 | % de muestras por árbol — junto con bagging_freq |
| `reg_lambda` | 1e-8–10 (log) | L2 regularization — controla magnitud de pesos |
| `reg_alpha` | 1e-8–10 (log) | L1 regularization — induce sparsity |
| `scale_pos_weight` | 10–RATIO (libre) | Peso del desbalance — clave para la ganancia |
| `max_depth` | 3–12 | Profundidad máxima — con constraint sobre num_leaves |

**Métrica objetivo de Optuna:** ganancia en el threshold óptimo (no AUC) — porque lo que maximizamos es exactamente la función de ganancia de la competencia.

In [ ]:
# Usamos fold_4 para optimizar (el más representativo del escenario real)
Xtr_opt, ytr_opt, Xte_opt, yte_opt = get_split(
    df, [202103, 202104, 202105], 202106, FEATURE_COLS, TARGET
)
yte_opt_arr = yte_opt.values

def objective_optuna(trial):
    max_depth = trial.suggest_int('max_depth', 3, 12)
    # num_leaves debe ser menor a 2^max_depth para evitar árboles degenerados
    num_leaves_max = min(256, 2**max_depth - 1)

    params = {
        'objective':         'binary',
        'metric':            'binary_logloss',
        'boosting_type':     'gbdt',
        'verbosity':        -1,
        'n_jobs':           -1,
        'random_state':      SEED,
        # ── Parámetros a optimizar ──────────────────────────────────────────
        'num_leaves':        trial.suggest_int('num_leaves', 16, num_leaves_max),
        'max_depth':         max_depth,
        'learning_rate':     trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'n_estimators':      trial.suggest_int('n_estimators', 100, 1000, step=50),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
        'feature_fraction':  trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction':  trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq':      trial.suggest_int('bagging_freq', 1, 7),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        # scale_pos_weight: exploramos desde la mitad del ratio natural hasta el máximo
        # A veces menos peso al positivo mejora la ganancia porque baja el threshold
        'scale_pos_weight':  trial.suggest_float('scale_pos_weight',
                                                   SCALE_POS_WEIGHT / 4,
                                                   SCALE_POS_WEIGHT * 1.5),
    }

    model = lgb.LGBMClassifier(**params)
    model.fit(
        Xtr_opt, ytr_opt,
        eval_set=[(Xte_opt, yte_opt)],
        callbacks=[lgb.early_stopping(30, verbose=False), lgb.log_evaluation(-1)]
    )

    prob = model.predict_proba(Xte_opt)[:, 1]
    opt  = threshold_optimo(yte_opt_arr, prob)
    return opt['ganancia']   # maximizamos ganancia directamente


print('Iniciando optimización Optuna...')
print('(Puede tomar 10-20 minutos en Colab según n_trials)')

N_TRIALS = 50   # Subir a 100-150 para producción — acá balanceamos tiempo/calidad

sampler = optuna.samplers.TPESampler(seed=SEED)
study = optuna.create_study(
    direction='maximize',
    sampler=sampler,
    study_name='lgbm_churn_ganancia'
)
study.optimize(objective_optuna, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\n✓ Optuna completado — {len(study.trials)} trials')
print(f'Mejor ganancia: ${study.best_value:,.0f}')
print(f'Mejores parámetros:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

In [ ]:
# ── Gráficos de Optuna ─────────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Historia de optimización
ax = axes[0]
valores = [t.value for t in study.trials if t.value is not None]
mejor_acum = pd.Series(valores).cummax()
ax.plot(valores, alpha=0.4, color='#4C72B0', label='Trial')
ax.plot(mejor_acum, color='#C44E52', lw=2, label='Mejor acumulado')
ax.axhline(0, color='gray', ls=':', lw=1)
ax.set_title('Historia de optimización Optuna', fontweight='bold')
ax.set_xlabel('Trial')
ax.set_ylabel('Ganancia ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1e9:.2f}B'))
ax.legend()

# Importancia de hiperparámetros
ax = axes[1]
try:
    importances = optuna.importance.get_param_importances(study)
    params_imp  = list(importances.keys())
    vals_imp    = list(importances.values())
    sorted_idx  = np.argsort(vals_imp)
    ax.barh([params_imp[i] for i in sorted_idx],
            [vals_imp[i]   for i in sorted_idx],
            color='#DD8452', edgecolor='white')
    ax.set_title('Importancia de hiperparámetros\n(contribución a la ganancia)', fontweight='bold')
    ax.set_xlabel('Importancia relativa')
except Exception:
    ax.text(0.5, 0.5, 'Requiere más trials\npara calcular importancia',
            ha='center', va='center', transform=ax.transAxes)

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de los mejores trials por parámetro
trials_df = study.trials_dataframe()
top_trials = trials_df.nlargest(10, 'value')
print('Top 10 trials por ganancia:')
cols_mostrar = ['number', 'value'] + [c for c in trials_df.columns if c.startswith('params_')]
print(top_trials[cols_mostrar].to_string(index=False))

In [ ]:
# Guardamos los mejores parámetros
mejores_params = study.best_params.copy()
mejores_params['ganancia_optuna_fold4'] = study.best_value
mejores_params['n_trials'] = N_TRIALS
mejores_params['fecha'] = datetime.now().isoformat()

with open(ARCHIVO_PARAMS, 'w') as f:
    json.dump(mejores_params, f, indent=2)
print(f'✓ Parámetros guardados en: {ARCHIVO_PARAMS}')

## 8 · Evaluación walk-forward completa

Entrenamos con los mejores parámetros sobre los 3 folds para evaluar estabilidad.
Si la ganancia sube consistentemente de F2 → F3 → F4, el modelo generaliza bien.

In [ ]:
# Construimos params finales combinando base + mejores de Optuna
PARAMS_OPT = {
    'objective':     'binary',
    'metric':        'binary_logloss',
    'boosting_type': 'gbdt',
    'verbosity':    -1,
    'n_jobs':       -1,
    'random_state':  SEED,
    **study.best_params
}

resultados_wf = []

for fold in FOLDS:
    print(f"\n{'='*55}")
    print(f"  {fold['nombre'].upper()} — test: {fold['test']}")
    print(f"  Train: {fold['train']}")
    print('='*55)

    Xtr, ytr, Xte, yte = get_split(df, fold['train'], fold['test'], FEATURE_COLS, TARGET)

    modelo = lgb.LGBMClassifier(**PARAMS_OPT)
    modelo.fit(
        Xtr, ytr,
        eval_set=[(Xte, yte)],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
    )

    prob = modelo.predict_proba(Xte)[:, 1]
    auc  = roc_auc_score(yte, prob)

    print(f'  AUC: {auc:.4f}')
    opt = plot_ganancia_threshold(yte.values, prob, titulo=f"({fold['nombre']})")

    resultados_wf.append({
        'fold':           fold['nombre'],
        'mes_test':       fold['test'],
        'n_train':        len(ytr),
        'n_test':         len(yte),
        'n_baja2_test':   int(yte.sum()),
        'auc':            round(auc, 4),
        'ganancia':       opt['ganancia'],
        'threshold_opt':  opt['threshold'],
        'tp':             opt['tp'],
        'fp':             opt['fp'],
        'n_intervenidos': opt['n_intervenidos'],
    })

    gc.collect()

resultados_df = pd.DataFrame(resultados_wf)
print(f'\n{"="*55}')
print('RESUMEN WALK-FORWARD')
print('='*55)
print(resultados_df.to_string(index=False))

In [ ]:
# Gráfico resumen walk-forward
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

folds_labels = resultados_df['fold'].tolist()

# Ganancia
ax = axes[0]
colores = ['#55A868' if g > 0 else '#C44E52' for g in resultados_df['ganancia']]
ax.bar(folds_labels, resultados_df['ganancia'] / 1e9, color=colores, edgecolor='white')
ax.set_title('Ganancia por fold', fontweight='bold')
ax.set_ylabel('Ganancia ($B)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x:.2f}B'))
ax.axhline(0, color='gray', ls='--', lw=1)

# AUC
ax = axes[1]
ax.bar(folds_labels, resultados_df['auc'], color='#4C72B0', edgecolor='white')
ax.set_title('AUC por fold', fontweight='bold')
ax.set_ylabel('AUC')
ax.set_ylim(0.5, 1.0)
for i, v in enumerate(resultados_df['auc']):
    ax.text(i, v + 0.005, f'{v:.4f}', ha='center', fontsize=10)

# Threshold óptimo
ax = axes[2]
ax.bar(folds_labels, resultados_df['threshold_opt'], color='#DD8452', edgecolor='white')
ax.set_title('Threshold óptimo por fold', fontweight='bold')
ax.set_ylabel('Threshold')
for i, v in enumerate(resultados_df['threshold_opt']):
    ax.text(i, v + 0.001, f'{v:.4f}', ha='center', fontsize=10)

plt.tight_layout()
plt.show()

# Guardamos métricas
resultados_df['timestamp'] = datetime.now().isoformat()
if ARCHIVO_METRICAS.exists():
    hist = pd.read_csv(ARCHIVO_METRICAS)
    resultados_df = pd.concat([hist, resultados_df], ignore_index=True)
resultados_df.to_csv(ARCHIVO_METRICAS, index=False)
print(f'✓ Métricas guardadas en: {ARCHIVO_METRICAS}')

## 9 · Análisis de errores — ¿dónde pifió el modelo?

En competencia no sabemos quiénes son los churners nuevos. Pero sobre los folds históricos sí podemos analizar:
- **Falsos negativos (FN):** churners que no detectamos — ¿qué perfil tienen?
- **Falsos positivos (FP):** clientes que intervenimos en vano — ¿son parecidos a los churners?

Este análisis sirve para guiar el feature engineering de la siguiente iteración.

In [ ]:
# Analizamos el fold_4 (el más relevante)
print('Analizando errores sobre fold_4 (test = 202106)...')

Xtr_f4, ytr_f4, Xte_f4, yte_f4 = get_split(
    df, [202103, 202104, 202105], 202106, FEATURE_COLS, TARGET
)

# Reentrenamos el modelo optimizado sobre fold_4
modelo_f4 = lgb.LGBMClassifier(**PARAMS_OPT)
modelo_f4.fit(
    Xtr_f4, ytr_f4,
    eval_set=[(Xte_f4, yte_f4)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)]
)

prob_f4 = modelo_f4.predict_proba(Xte_f4)[:, 1]
opt_f4  = threshold_optimo(yte_f4.values, prob_f4)
pred_f4 = (prob_f4 >= opt_f4['threshold']).astype(int)

# Construimos el dataframe de test con scores
df_test_f4 = df[(df[COL_PERIODO] == 202106) & df[TARGET].notna()].copy()
df_test_f4 = df_test_f4.reset_index(drop=True)
df_test_f4['prob']  = prob_f4
df_test_f4['pred']  = pred_f4
df_test_f4['real']  = yte_f4.values
df_test_f4['error'] = 'TN'
df_test_f4.loc[(df_test_f4['real']==1) & (df_test_f4['pred']==1), 'error'] = 'TP'
df_test_f4.loc[(df_test_f4['real']==0) & (df_test_f4['pred']==1), 'error'] = 'FP'
df_test_f4.loc[(df_test_f4['real']==1) & (df_test_f4['pred']==0), 'error'] = 'FN'

print('\nMatriz de confusión (fold_4):')
print(df_test_f4['error'].value_counts())
print(f"\nGanancia con threshold óptimo: ${opt_f4['ganancia']:,.0f}")

In [ ]:
# Comparación de perfil: TP vs FN (churners detectados vs perdidos)
# Usamos variables originales clave para entender el perfil
VARS_PERFIL = [
    'cliente_edad', 'cliente_antiguedad', 'mrentabilidad',
    'cproductos', 'mcuentas_saldo', 'chomebanking_transacciones',
    'mcomisiones', 'prob'
]
VARS_PERFIL = [v for v in VARS_PERFIL if v in df_test_f4.columns]

# Solo churners reales: TP vs FN
churners = df_test_f4[df_test_f4['real'] == 1]
perfil_tp = churners[churners['error'] == 'TP'][VARS_PERFIL].describe().T[['mean', '50%']]
perfil_fn = churners[churners['error'] == 'FN'][VARS_PERFIL].describe().T[['mean', '50%']]

perfil_comp = pd.concat(
    [perfil_tp.add_suffix('_TP'), perfil_fn.add_suffix('_FN')], axis=1
)
perfil_comp['diferencia_%'] = (
    (perfil_comp['mean_FN'] - perfil_comp['mean_TP']) / perfil_comp['mean_TP'].abs() * 100
).round(1)

print('Perfil: churners detectados (TP) vs perdidos (FN)')
print(perfil_comp.round(2))

In [ ]:
# Distribución de probabilidades por tipo de error
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Hist de scores por tipo de error
ax = axes[0]
for error_type, color in [('TP', '#55A868'), ('FP', '#DD8452'), ('FN', '#C44E52'), ('TN', '#4C72B0')]:
    subset = df_test_f4[df_test_f4['error'] == error_type]['prob']
    if len(subset) > 0:
        ax.hist(subset, bins=40, alpha=0.5, label=f'{error_type} (n={len(subset):,})', color=color)
ax.axvline(opt_f4['threshold'], color='black', ls='--', lw=1.5,
           label=f"Threshold: {opt_f4['threshold']:.4f}")
ax.set_title('Distribución de scores por tipo de error', fontweight='bold')
ax.set_xlabel('Probabilidad predicha (baja2)')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=8)
ax.set_yscale('log')

# Score de FN vs TP
ax = axes[1]
for error_type, color, label in [
    ('TP', '#55A868', 'Detectados (TP)'),
    ('FN', '#C44E52', 'Perdidos (FN)')
]:
    subset = churners[churners['error'] == error_type]['prob']
    if len(subset) > 0:
        ax.hist(subset, bins=30, alpha=0.6, label=f'{label} (n={len(subset)})', color=color)
ax.axvline(opt_f4['threshold'], color='black', ls='--', lw=1.5)
ax.set_title('Score de churners: detectados vs perdidos', fontweight='bold')
ax.set_xlabel('Probabilidad predicha (baja2)')
ax.set_ylabel('Frecuencia')
ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 10 · Predicción final sobre 202108

Entrenamos con **todos los meses disponibles** (202103–202107) para predecir 202108.

El threshold a usar es el **promedio de thresholds óptimos** de los 3 folds — más robusto que usar solo uno. Si los folds muestran thresholds muy distintos, conviene usar el del fold más reciente (fold_4).

**Nota sobre 202107:** tiene `sin_target` pero las features del mes (saldos, transacciones, comportamiento) son reales — lo incluimos en el train final para que el modelo vea el estado más reciente de los clientes.

In [ ]:
# Threshold para la predicción final
thresholds_folds = resultados_df['threshold_opt'].values
threshold_final = float(np.mean(thresholds_folds))

print('Thresholds por fold:')
for i, (fold, t) in enumerate(zip(FOLDS, thresholds_folds)):
    print(f"  {fold['nombre']}: {t:.5f}")
print(f'\nThreshold promedio (a usar): {threshold_final:.5f}')
print(f'Threshold fold_4 (alternativo): {thresholds_folds[-1]:.5f}')

In [ ]:
# Train final: todos los meses CON target real (202103–202106)
# También podemos incluir 202107 como pseudo-train
# pero sus features son las del mes, no tiene target -> lo usamos solo en features

mask_train_final = df[COL_PERIODO].isin(MESES_CON_TARGET) & df[TARGET].notna()
X_train_final = df.loc[mask_train_final, FEATURE_COLS]
y_train_final = df.loc[mask_train_final, TARGET].astype(int)

# Test final: 202108
mask_pred_final = df[COL_PERIODO] == MES_PREDECIR
X_pred_final    = df.loc[mask_pred_final, FEATURE_COLS]
ids_pred_final  = df.loc[mask_pred_final, COL_ID]

print(f'Train final: {len(y_train_final):,} filas (baja2={y_train_final.sum():,})')
print(f'Clientes a predecir: {len(X_pred_final):,}')

# Entrenamos modelo final con parámetros optimizados
params_finales = {k: v for k, v in PARAMS_OPT.items()}

modelo_final = lgb.LGBMClassifier(**params_finales)
modelo_final.fit(X_train_final, y_train_final, callbacks=[lgb.log_evaluation(-1)])

prob_final = modelo_final.predict_proba(X_pred_final)[:, 1]
pred_final = (prob_final >= threshold_final).astype(int)

print(f'\nResultado de predicción (threshold={threshold_final:.5f}):')
print(f'  Clientes a intervenir (pred=1): {pred_final.sum():,}')
print(f'  Clientes no intervenir (pred=0): {(pred_final==0).sum():,}')
print(f'  % intervenidos: {pred_final.mean()*100:.2f}%')

In [ ]:
# Distribución de scores en la predicción final
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(prob_final, bins=60, color='#4C72B0', edgecolor='white', alpha=0.85)
ax.axvline(threshold_final, color='#C44E52', ls='--', lw=2,
           label=f'Threshold: {threshold_final:.4f}')
ax.set_yscale('log')
ax.set_title('Distribución de scores — 202108 (a predecir)', fontweight='bold')
ax.set_xlabel('Probabilidad predicha (baja2)')
ax.set_ylabel('N clientes (escala log)')
ax.legend()

ax = axes[1]
ax.hist(prob_final[prob_final >= threshold_final], bins=40,
        color='#C44E52', edgecolor='white', alpha=0.85)
ax.set_title('Scores de clientes a intervenir (pred=1)', fontweight='bold')
ax.set_xlabel('Probabilidad predicha (baja2)')
ax.set_ylabel('N clientes')

plt.tight_layout()
plt.show()

In [ ]:
# ── Generamos el archivo de submission ────────────────────────────────────────
# Formato: numero_de_cliente, Predicted (1 = intervenir, 0 = no intervenir)
# El archivo incluye TODOS los clientes del mes a predecir

submission = pd.DataFrame({
    COL_ID:       ids_pred_final.values,
    'Predicted':  pred_final,
    'prob_baja2': prob_final.round(6),   # guardamos la proba por si queremos probar otro threshold
})

submission.to_csv(ARCHIVO_SUBMISSION, index=False)

print(f'✓ Submission guardado en: {ARCHIVO_SUBMISSION}')
print(f'  Total filas: {len(submission):,}')
print(f'  Pred=1 (intervenir): {submission["Predicted"].sum():,}')
print(f'  Pred=0 (no intervenir): {(submission["Predicted"]==0).sum():,}')
print()
print('Primeras filas del submission:')
print(submission.head(10))

## 11 · Resumen del experimento

In [ ]:
print('=' * 65)
print('RESUMEN DEL EXPERIMENTO')
print('=' * 65)
print(f'Fecha: {datetime.now().strftime("%Y-%m-%d %H:%M")}')
print()
print('FEATURE ENGINEERING')
print(f'  Features originales: {len(cols_base_fe)}')
print(f'  Features totales:    {len(FEATURE_COLS)}')
print(f'  (lags, deltas, deltas rel., tendencias, ratios, flags)')
print()
print('OPTUNA')
print(f'  Trials: {N_TRIALS}')
print(f'  Mejor ganancia (fold_4): ${study.best_value:,.0f}')
print(f'  Mejores parámetros clave:')
for k in ['num_leaves', 'learning_rate', 'min_child_samples', 'scale_pos_weight']:
    if k in study.best_params:
        print(f'    {k}: {study.best_params[k]}')
print()
print('WALK-FORWARD')
for _, row in resultados_df.iterrows():
    if 'timestamp' not in row or pd.isna(row.get('timestamp', None)):
        continue
    g = row.get('ganancia', 0)
    print(f"  {row['fold']}: ganancia=${g:>15,.0f} | AUC={row.get('auc', 0):.4f} | threshold={row.get('threshold_opt', 0):.4f}")
print()
print('PREDICCIÓN FINAL (202108)')
print(f'  Threshold usado: {threshold_final:.5f}')
print(f'  Clientes a intervenir: {pred_final.sum():,} ({pred_final.mean()*100:.2f}%)')
print()
print('ARCHIVOS GENERADOS')
print(f'  Submission:       {ARCHIVO_SUBMISSION.name}')
print(f'  Métricas:         {ARCHIVO_METRICAS.name}')
print(f'  Params Optuna:    {ARCHIVO_PARAMS.name}')

In [ ]:
# Limpieza final
del df, X_train_final, y_train_final, X_pred_final
del Xtr_f4, ytr_f4, Xte_f4, yte_f4, df_test_f4
gc.collect()
print('✓ Memoria liberada. Notebook completada.')